# Notebook 05: Feature Engineering & Preprocessing for Crash Severity Prediction
## Addis Ababa Road Traffic Accident Safety Analysis

**Author:** YonInsights (`yo2120018@gmail.com`)  
**Curriculum Alignment:** ALX Data Science & Machine Learning — Supervised Learning & Preprocessing  
**Dataset Scale:** 13,063 collision dockets | 31 attributes  

---

### Analytical Philosophy & Problem Formulation:
In road safety management, accurate predictive models assist emergency response triage and proactive infrastructure investments. However, real-world crash dockets present severe data science hurdles:
1. **Extreme Class Imbalance:** 82.5% Slight Injury vs. 15.1% Serious vs. 2.4% Fatal. A naive majority-class predictor achieves 82.5% accuracy while completely failing to detect life-threatening events.
2. **Data Leakage Vulnerability:** Attributes recorded *after* a collision occurs (such as post-impact casualty demographics) must never be used to predict pre-crash severity.
3. **Pseudo-Missing Data & Entry Noise:** Literal text placeholders (`'na'`, `'Unknown'`) and numeric form artifacts (`'7'`, `'5'`) must be cleansed without corrupting statistical validity.
4. **Leakage-Free Transformations:** Standard scaling and one-hot encoding parameters must be learned **strictly on the training split** and applied to test dockets without data snooping.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

# Scikit-learn preprocessing and model selection tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Set reproducibility seed
RANDOM_STATE = 42

# Define paths
DATA_PATH = Path("../Data/Addis_Ababa_city_RTA.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("Data/Addis_Ababa_city_RTA.csv")

MODEL_DIR = Path("../models")
if not MODEL_DIR.exists():
    MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Ingest raw dataset
df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset successfully loaded. Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")


Dataset successfully loaded. Shape: 13,063 rows x 31 columns


---

## Step 1: Target Variable Formulation & Addressing Class Imbalance

### The Business & Road Safety Problem:
In emergency management, the priority is distinguishing **high-severity collisions** (which require immediate trauma surgical dispatch, life-support transit, and rapid accident reconstruction) from **slight collisions** (which can be handled by standard roadside protocol).

Furthermore, in the raw 3-class distribution:
- `Slight Injury`: 10,776 (82.5%)
- `Serious Injury`: 1,976 (15.1%)
- `Fatal injury`: 311 (2.4%)

The 2.4% fatal class creates extreme sample sparsity. By collapsing outcomes into a binary indicator:
- **Class 1 (Severe / Fatal):** Serious Injury + Fatal injury (2,287 records, **17.51%**)
- **Class 0 (Slight):** Slight Injury (10,776 records, **82.49%**)

This formulation aligns directly with municipal intervention priorities while preserving a robust sample size for minority class learning.


In [2]:
# Create binary classification target: 1 = Severe/Fatal, 0 = Slight Injury
df = df_raw.copy()
df['is_severe'] = df['Accident_severity'].isin(['Serious Injury', 'Fatal injury']).astype(int)

# Validate class frequencies
target_counts = df['is_severe'].value_counts()
target_props = df['is_severe'].value_counts(normalize=True) * 100

summary_target = pd.DataFrame({
    'Collision Count': target_counts,
    'Proportion (%)': target_props.round(2)
}, index=[0, 1])
summary_target.index.name = 'Class (is_severe)'
summary_target.rename(index={0: '0: Slight Injury', 1: '1: Serious or Fatal Injury'}, inplace=True)

print("Target Class Distribution:")
print(summary_target)


Target Class Distribution:
                            Collision Count  Proportion (%)
Class (is_severe)                                          
0: Slight Injury                      10776           82.49
1: Serious or Fatal Injury             2287           17.51


---

## Step 2: Feature Engineering & Eliminating Data Leakage

### What is Data Leakage?
Data leakage occurs when information from outside the training dataset or information that would **not be available at inference time** is included in the feature set.

### 🚫 Features Excluded to Prevent Post-Crash Outcome Leakage:
1. `Casualty_class`, `Casualty_gender`, `Age_band_of_casualty`, `Work_of_casuality`, `Fitness_of_casuality`:
   - These represent post-collision hospital or triage evaluations.
   - In 36.2% of cases, they contain the placeholder `'na'` because no secondary casualty was interviewed. Including them introduces severe target-correlated missingness.
2. `Number_of_casualties`:
   - Evaluated as a collision outcome rather than a pre-collision environmental or behavioral predictor.

### ✅ Features Retained (Available at or Before Impact):
- **Temporal:** `Hour` (extracted from `Time`), `Day_of_week`
- **Driver Characteristics:** `Age_band_of_driver`, `Drivers_gender`, `Educational_level`, `Driving_experience`, `Vehicle_driver_relation`
- **Vehicle Characteristics:** `Type_of_vehicle`, `Owner_of_vehicle`, `Service_year_of_vehicle`, `Defect_of_vehicle`, `Number_of_vehicles_involved`
- **Road & Environmental Infrastructure:** `Area_accident_occured`, `Lanes_or_Medians`, `Road_allignment`, `Types_of_Junction`, `Road_surface_type`, `Road_surface_conditions`, `Light_conditions`, `Weather_conditions`
- **Dynamic Context:** `Type_of_collision`, `Vehicle_movement`, `Pedestrian_movement`, `Cause_of_accident`


In [3]:
# 1. Feature Engineering: Extract 24-hour integer clock from Time
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour

# 2. Define quantitative (numeric) predictors
numeric_features = [
    'Hour',
    'Number_of_vehicles_involved'
]

# 3. Define nominal (categorical) predictors
categorical_features = [
    'Day_of_week',
    'Age_band_of_driver',
    'Drivers_gender',
    'Educational_level',
    'Vehicle_driver_relation',
    'Driving_experience',
    'Type_of_vehicle',
    'Owner_of_vehicle',
    'Service_year_of_vehicle',
    'Defect_of_vehicle',
    'Area_accident_occured',
    'Lanes_or_Medians',
    'Road_allignment',
    'Types_of_Junction',
    'Road_surface_type',
    'Road_surface_conditions',
    'Light_conditions',
    'Weather_conditions',
    'Type_of_collision',
    'Vehicle_movement',
    'Pedestrian_movement',
    'Cause_of_accident'
]

# 4. Standardize text placeholders and unmapped docket codes
for col in categorical_features:
    df[col] = df[col].fillna('Unknown').astype(str).str.strip()
    df[col] = df[col].replace({
        'na': 'Unknown',
        'nan': 'Unknown',
        'NaN': 'Unknown',
        '5': 'Unknown',
        '7': 'Unknown'
    })

# 5. Isolate feature matrix X and target vector y
feature_cols = numeric_features + categorical_features
X = df[feature_cols].copy()
y = df['is_severe'].copy()

print(f"Feature matrix isolated: {X.shape[0]:,} samples across {len(feature_cols)} predictors ({len(numeric_features)} numeric, {len(categorical_features)} categorical).")


Feature matrix isolated: 13,063 samples across 24 predictors (2 numeric, 22 categorical).


---

## Step 3: Stratified Train-Test Splitting

### Why Stratification is Mandatory for Imbalanced Data:
If we apply standard unstratified random sampling to an 82.5% vs. 17.5% imbalanced dataset, stochastic fluctuation can result in unequal severe crash proportions between partitions.

By setting `stratify=y`, scikit-learn partitions both the majority class ($y=0$) and minority class ($y=1$) proportionately, ensuring that both training (80%) and testing (20%) sets contain an exact **17.51%** severe crash ratio.


In [4]:
# Partition into 80% training and 20% holdout testing sets with exact target stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=RANDOM_STATE, 
    stratify=y
)

# Verify split dimensions and class proportions
split_audit = pd.DataFrame({
    'Train Count': y_train.value_counts(),
    'Train Proportion (%)': (y_train.value_counts(normalize=True) * 100).round(2),
    'Test Count': y_test.value_counts(),
    'Test Proportion (%)': (y_test.value_counts(normalize=True) * 100).round(2)
}, index=[0, 1])
split_audit.index.name = 'Target Class (is_severe)'
split_audit.rename(index={0: '0: Slight', 1: '1: Severe/Fatal'}, inplace=True)

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")
print(split_audit)


X_train shape: (10450, 24) | X_test shape: (2613, 24)
                          Train Count  ...  Test Proportion (%)
Target Class (is_severe)               ...                     
0: Slight                        8620  ...                82.51
1: Severe/Fatal                  1830  ...                17.49

[2 rows x 4 columns]


---

## Step 4: Modular Scikit-Learn Preprocessing Pipeline

### Preventing Preprocessing Leakage:
A classic mistake in machine learning is calculating the mean and standard deviation (or identifying unique categorical levels) across the *entire* dataset before splitting. This leaks test set distributions into training!

To guarantee strict leakage isolation:
1. **Numerical Pipeline:** Impute missing values via `SimpleImputer(strategy='median')` and standardize via `StandardScaler()`.
2. **Categorical Pipeline:** Impute missing categories via `SimpleImputer(strategy='constant', fill_value='Unknown')` and encode via `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`.
3. Combine using `ColumnTransformer` and fit **exclusively** on `X_train`.
4. Apply the fitted pipeline to transform `X_test`.


In [5]:
# 1. Define sub-pipelines for numerical and categorical pathways
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 2. Assemble unified ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features),
        ('cat', cat_pipeline, categorical_features)
    ],
    remainder='drop'
)

# 3. Fit STRICTLY on X_train, then transform both splits
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 4. Extract generated feature names
encoded_feature_names = preprocessor.get_feature_names_out()

print("Preprocessing Pipeline Executed Successfully!")
print(f"Processed X_train shape: {X_train_processed.shape} ({len(encoded_feature_names)} engineered features)")
print(f"Processed X_test shape:  {X_test_processed.shape}")

# 5. Persist preprocessor and preprocessed arrays to models directory
joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")
np.save(MODEL_DIR / "X_train_processed.npy", X_train_processed)
np.save(MODEL_DIR / "X_test_processed.npy", X_test_processed)
np.save(MODEL_DIR / "y_train.npy", y_train.values)
np.save(MODEL_DIR / "y_test.npy", y_test.values)

print(f"\nArtifacts persisted to '{MODEL_DIR.resolve()}':")
print("  - preprocessor.joblib")
print("  - X_train_processed.npy")
print("  - X_test_processed.npy")
print("  - y_train.npy")
print("  - y_test.npy")


Preprocessing Pipeline Executed Successfully!
Processed X_train shape: (10450, 179) (179 engineered features)
Processed X_test shape:  (2613, 179)

Artifacts persisted to 'D:\Projects\Addis-Ababa-Road-Safety-Analysis\models':
  - preprocessor.joblib
  - X_train_processed.npy
  - X_test_processed.npy
  - y_train.npy
  - y_test.npy


---

## Step 5: Summary, Takeaways & Concept Check

### What We Have Accomplished in Preprocessing:
1. **Target Formulation:** Converted a 3-class target with extreme 2.4% fatal sparsity into an operationally actionable binary target (`0: Slight` vs. `1: Severe/Fatal`).
2. **Strict Data Leakage Prevention:** Discarded post-crash triage attributes (`Casualty_class`, `Age_band_of_casualty`) that would introduce artificial target leakage or missingness bias.
3. **Pseudo-Missing Sanitation:** Mapped all `'na'`, `'nan'`, and docket code artifacts (`'7'`, `'5'`) to explicit `'Unknown'` categories.
4. **Stratified Split:** Split 13,063 records into 10,450 training and 2,613 testing records, preserving the exact 17.51% severe crash ratio in both.
5. **Production Transformation:** Scaled quantitative continuous features and one-hot encoded nominal categories into **179 binary feature columns**, saving the reproducible pipeline artifact.

---

### 🧠 Check Your Understanding (Pedagogical Reflection):
1. **Why must we fit the `StandardScaler` and `OneHotEncoder` only on `X_train` rather than the whole dataset?**  
   *Answer:* If we fit on the full dataset, the mean, variance, and category list from the test set leak into training, artificially inflating validation metrics and leading to unexpected failure in production.
2. **Why is overall classification accuracy a dangerous metric for this dataset?**  
   *Answer:* A dummy classifier that predicts "Slight" for every single crash achieves **82.5% accuracy**, but its recall on fatal and serious crashes is **0%**—meaning every single life-threatening incident is missed!
